# Urban Biotope Semantic Segmentation with SegFormer

This notebook trains a **semantic segmentation model** to classify **biotope types in Zurich** using high-resolution aerial imagery (SWISSIMAGE 10 cm).

The workflow:

1. Load dataset from Google Drive
2. Prepare PyTorch dataset
3. Apply preprocessing and augmentations
4. Fine-tune a SegFormer model
5. Evaluate performance
6. Visualize predictions

We use a **transformer-based segmentation model (SegFormer)** from Hugging Face.


## Setup

#### Install Dependencies


In [1]:
!pip install -q transformers datasets evaluate accelerate
!pip install -q rasterio albumentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.6 MB/s eta 0:00:00


#### Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Set datapath:

In [3]:
DATA_ROOT = "/content/drive/MyDrive/subset"


#### Check Runtime
Check if T4 runtime is working correctly

In [4]:
import torch

print("GPU:", torch.cuda.is_available())


GPU: True


# Modeling

## Import Libraries

In [5]:
import os
import numpy as np
import torch
import rasterio
from PIL import Image

from torch.utils.data import Dataset, DataLoader

from transformers import (
    SegformerForSemanticSegmentation,
    SegformerImageProcessor,
    TrainingArguments,
    Trainer
)

import albumentations as A


## Define Class Labels

In [6]:
id2label = {
0: "Impervious / Built-up",
1: "Forest & Dense Canopy",
2: "Urban Trees & Bushes",
3: "Managed Green Spaces",
4: "Standard Grassland",
5: "Ecologically Valuable Meadow",
6: "Agriculture & Orchards",
7: "Water Bodies"
}

label2id = {v:k for k,v in id2label.items()}
num_classes = len(id2label)


## Dataset Structure

Each datapoint consists of:

Image patch  
- RGB satellite image  
- size: 1024 × 1024  

Mask patch  
- same spatial resolution  
- each pixel contains a **class ID (0–7)**  
- 255 represents **ignore / unlabeled**

During training we randomly crop patches to **512×512**, which improves
generalization and fits GPU memory.


## Dataset Class

In [7]:
class BiotopeDataset(Dataset):

    def __init__(self, root_dir, split, processor, transform=None):

        self.img_dir = os.path.join(root_dir, split, "images")
        self.mask_dir = os.path.join(root_dir, split, "masks")

        self.images = sorted(os.listdir(self.img_dir))
        self.processor = processor
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_path = os.path.join(self.img_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx])

        with rasterio.open(img_path) as src:
            image = src.read([1,2,3])
            image = np.transpose(image, (1,2,0))

        with rasterio.open(mask_path) as src:
            mask = src.read(1)

        if self.transform:

            augmented = self.transform(
                image=image,
                mask=mask
            )

            image = augmented["image"]
            mask = augmented["mask"]

        encoded = self.processor(
            images=image,
            segmentation_maps=mask,
            return_tensors="pt"
        )

        encoded = {k:v.squeeze() for k,v in encoded.items()}

        return encoded


## Data Augmentation

Satellite imagery benefits strongly from augmentation.

We apply:

- random crop
- horizontal flip
- vertical flip
- rotation

This increases robustness and reduces overfitting.


In [8]:
train_transform = A.Compose([
    A.RandomCrop(512,512),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5)
])

val_transform = A.Compose([
    A.CenterCrop(512,512)
])


## Load Model

We use **SegFormer-B3**, a transformer-based semantic segmentation model.

Advantages:

- strong performance on high-resolution imagery
- efficient transformer architecture
- robust multi-scale feature extraction


In [9]:
processor = SegformerImageProcessor.from_pretrained(
    "nvidia/segformer-b3-finetuned-ade-512-512"
)

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b3-finetuned-ade-512-512",
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:417: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/190M [00:00<?, ?B/s]

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b3-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.weight: found shape torch.Size([150, 768, 1, 1]) in the checkpoint and torch.Size([8, 768, 1, 1]) in the model instantiated
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([8]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/189M [00:00<?, ?B/s]

## Load Dataset

In [ ]:
train_dataset = BiotopeDataset(
    DATA_ROOT,
    "train",
    processor,
    transform=train_transform
)

val_dataset = BiotopeDataset(
    DATA_ROOT,
    "val",
    processor,
    transform=val_transform
)


TypeError: join() argument must be str, bytes, or os.PathLike object, not 'list'

#### Subset Train and Val Dataset for Testing

In [ ]:
from torch.utils.data import Subset
import numpy as np

SUBSET_RATIO = 0.1  # use 10% of the dataset

train_size = int(len(train_dataset) * SUBSET_RATIO)
val_size = int(len(val_dataset) * SUBSET_RATIO)

train_indices = np.random.choice(len(train_dataset), train_size, replace=False)
val_indices = np.random.choice(len(val_dataset), val_size, replace=False)

train_dataset = Subset(train_dataset, train_indices)
val_dataset = Subset(val_dataset, val_indices)

print("Train subset:", len(train_dataset))
print("Val subset:", len(val_dataset))


## Training Configuration

We fine-tune the pretrained SegFormer model.

Key parameters:

Batch size: small (T4 GPU memory)  
Learning rate: 6e-5 (standard for transformers)  
Epochs: 10–20 recommended


In [ ]:
# training_args = TrainingArguments(

#     output_dir="/content/segformer-biotope",

#     learning_rate=6e-5,

#     per_device_train_batch_size=2,
#     per_device_eval_batch_size=2,

#     num_train_epochs=15,

#     eval_strategy="epoch",
#     save_strategy="epoch",

#     logging_steps=50,

#     fp16=True,

#     load_best_model_at_end=True
# )


## Define Trainer

In [ ]:
# trainer = Trainer(

#     model=model,

#     args=training_args,

#     train_dataset=train_dataset,
#     eval_dataset=val_dataset
# )


## Train Model

Fine-tuning the SegFormer model on Zurich biotope data.


In [ ]:
# trainer.train()


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

## Training Loop

In [ ]:
# -------- DATALOADERS --------

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=2)

print(len(train_loader.dataset))

TypeError: join() argument must be str, bytes, or os.PathLike object, not 'list'

In [ ]:
# =====================
# CUSTOM TRAINING LOOP 
# =====================

import torch
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# -------- DATALOADERS --------

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=2)

# -------- OPTIMIZER --------

optimizer = torch.optim.AdamW(model.parameters(), lr=6e-5)

# -------- METRIC: mIoU --------

def compute_miou(preds, labels, num_classes=8):

    ious = []

    preds = preds.flatten()
    labels = labels.flatten()

    valid = labels != 255
    preds = preds[valid]
    labels = labels[valid]


    for cls in range(num_classes):

        pred_mask = preds == cls
        label_mask = labels == cls

        intersection = (pred_mask & label_mask).sum()
        union = (pred_mask | label_mask).sum()

        if union == 0:
            continue

        iou = intersection / union
        ious.append(iou)

    return np.mean(ious) if len(ious) > 0 else 0

# -------- TRAINING SETTINGS --------

num_epochs = 1
early_stop_miou = 0.65   # stop if reached
best_val_miou = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "train_miou": [],
    "val_miou": []
}

# ============================================================
# TRAIN LOOP
# ============================================================

for epoch in range(num_epochs):

    print(f"\n🚀 Epoch {epoch+1}/{num_epochs}")

    # ================= TRAIN =================
    model.train()

    train_loss = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(train_loader, desc="Training"):

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        import torch.nn.functional as F

        logits = outputs.logits

        # Upsample to match label size
        logits = F.interpolate(
            logits,
            size=labels.shape[-2:],   # (H, W)
            mode="bilinear",
            align_corners=False
        )

        preds = logits.argmax(dim=1).cpu().numpy()

        #preds = outputs.logits.argmax(dim=1).detach().cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels.cpu().numpy())

    train_loss /= len(train_loader)

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    train_miou = compute_miou(all_preds, all_labels)

    # ================= VALIDATION =================
    model.eval()

    val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):

            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss

            val_loss += loss.item()

            

            logits = outputs.logits

            # Upsample to match label size
            logits = F.interpolate(
                logits,
                size=labels.shape[-2:],   # (H, W)
                mode="bilinear",
                align_corners=False
            )

            preds = logits.argmax(dim=1).cpu().numpy()

            #preds = outputs.logits.argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())

    val_loss /= len(val_loader)

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    val_miou = compute_miou(all_preds, all_labels)

    # ================= LOGGING =================

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_miou"].append(train_miou)
    history["val_miou"].append(val_miou)

    print(f"\nEpoch {epoch+1} Results:")
    print(f"Train Loss: {train_loss:.4f} | Train mIoU: {train_miou:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   mIoU: {val_miou:.4f}")

    # ================= EARLY STOPPING =================

    if val_miou > best_val_miou:
        best_val_miou = val_miou
        torch.save(model.state_dict(), "/content/best_model.pth")
        print("✅ Best model saved!")

    if val_miou >= early_stop_miou:
        print(f"\n🎯 Early stopping: reached mIoU {val_miou:.4f}")
        break

print("\n🏁 Training complete!")



🚀 Epoch 1/1


Training: 100%|██████████| 520/520 [17:51<00:00,  2.06s/it]


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(12,5))

# Loss
plt.subplot(1,2,1)
plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Val Loss")
plt.legend()
plt.title("Loss")

# mIoU
plt.subplot(1,2,2)
plt.plot(epochs, history["train_miou"], label="Train mIoU")
plt.plot(epochs, history["val_miou"], label="Val mIoU")
plt.legend()
plt.title("mIoU")

plt.show()


In [ ]:
# ============================================================
# OVERFIT DEBUG VISUALIZATION (TRAIN vs VAL)
# ============================================================

import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn.functional as F

model.eval()

def get_prediction(sample):

    pixel_values = sample["pixel_values"].unsqueeze(0).to(device)
    labels = sample["labels"].cpu().numpy()

    with torch.no_grad():
        outputs = model(pixel_values=pixel_values)

    logits = outputs.logits

    # Upsample to match label size
    logits = F.interpolate(
        logits,
        size=labels.shape,
        mode="bilinear",
        align_corners=False
    )

    pred = logits.argmax(dim=1).cpu().numpy()[0]

    image = sample["pixel_values"].cpu().numpy()
    image = np.transpose(image, (1,2,0))
    image = (image - image.min()) / (image.max() - image.min())

    return image, labels, pred


# -------- GET ONE TRAIN + ONE VAL SAMPLE --------

train_sample = train_dataset[0]   # should overfit here
val_sample   = val_dataset[0]

train_img, train_gt, train_pred = get_prediction(train_sample)
val_img, val_gt, val_pred       = get_prediction(val_sample)

# -------- COLOR MAP --------

from matplotlib.colors import ListedColormap

colors = [
    (0.6, 0.6, 0.6),  # built-up
    (0.0, 0.4, 0.0),  # forest
    (0.2, 0.7, 0.2),  # trees
    (0.4, 0.9, 0.4),  # managed
    (0.8, 1.0, 0.4),  # grass
    (1.0, 0.8, 0.2),  # meadow
    (0.7, 0.5, 0.2),  # agriculture
    (0.2, 0.4, 1.0)   # water
]
cmap = ListedColormap(colors)

# -------- PLOT --------

plt.figure(figsize=(15,10))

# --- TRAIN ROW ---
plt.subplot(2,3,1)
plt.title("Train Image")
plt.imshow(train_img)
plt.axis("off")

plt.subplot(2,3,2)
plt.title("Train Ground Truth")
plt.imshow(train_gt, cmap=cmap, vmin=0, vmax=7)
plt.axis("off")

plt.subplot(2,3,3)
plt.title("Train Prediction")
plt.imshow(train_pred, cmap=cmap, vmin=0, vmax=7)
plt.axis("off")

# --- VAL ROW ---
plt.subplot(2,3,4)
plt.title("Val Image")
plt.imshow(val_img)
plt.axis("off")

plt.subplot(2,3,5)
plt.title("Val Ground Truth")
plt.imshow(val_gt, cmap=cmap, vmin=0, vmax=7)
plt.axis("off")

plt.subplot(2,3,6)
plt.title("Val Prediction")
plt.imshow(val_pred, cmap=cmap, vmin=0, vmax=7)
plt.axis("off")

plt.tight_layout()
plt.show()


## Save Model


In [ ]:
trainer.save_model("/content/drive/MyDrive/segformer-biotope-model")


## Prediction Example


In [ ]:
import matplotlib.pyplot as plt

sample = val_dataset[0]

with torch.no_grad():

    outputs = model(
        pixel_values=sample["pixel_values"].unsqueeze(0).cuda()
    )

pred = outputs.logits.argmax(dim=1).cpu().numpy()[0]

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.title("Ground Truth")
plt.imshow(sample["labels"])

plt.subplot(1,2,2)
plt.title("Prediction")
plt.imshow(pred)

plt.show()
